In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 🌊 Flood Segmentation Phase 2 — v3 OPTIMIZED
### ANRF Flood Prediction Challenge · West Bengal 2024
**Target: Top-5 from Rank 59**

**Key improvements over v2.1:**
- ✅ **CRITICAL BUG FIX**: TestDataset now uses `rasterio` + same `normalize_image()` as train (was using `cv2.imread` + `/255` — completely wrong for 6-channel SAR TIFs)
- ✅ **3-model ensemble**: added Segformer-B3 (transformer backbone) for diversity
- ✅ **Spectral indices**: NDWI + MNDWI as channels 7-8 (8 total)
- ✅ **SAR ratio**: HH/HV channel (detects flooded vegetation vs open water)
- ✅ **Longer training**: 60 epochs with OneCycleLR
- ✅ **Stratified K-Fold**: sample-level flood/noflood ratio stratification
- ✅ **Morphological post-processing**: remove tiny noise islands, fill holes
- ✅ **Flood threshold tuning** on validation set (default argmax is suboptimal)
- ✅ **Pseudo-labeling**: second pass with high-confidence test predictions

## 1. Install Dependencies

In [ ]:
!pip install -q segmentation-models-pytorch albumentations rasterio timm transformers
!pip install -q 'git+https://github.com/qubvel/segmentation_models.pytorch'

## 2. Imports & Seed

In [ ]:
# ══════════════════════════════════════════
#  CELL 1 — Run this after EVERY restart
#  then use Run All below, or run cells in order
# ══════════════════════════════════════════
import os, gc, math, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import rasterio
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import segmentation_models_pytorch as smp
import cv2
from scipy import ndimage

warnings.filterwarnings('ignore')
os.environ.setdefault('GDAL_DISABLE_READDIR_ON_OPEN', 'EMPTY_DIR')
os.environ.setdefault('CPL_VSIL_CURL_ALLOWED_EXTENSIONS', '.tif')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

print('Imports done ✅')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 3. Config

In [ ]:
class CFG:
    # ── Paths ──────────────────────────────────────────────────────────────
    BASE_PATH  = '/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data'
    TRAIN_TXT  = f'{BASE_PATH}/split/train.txt'
    VAL_TXT    = f'{BASE_PATH}/split/val.txt'
    TEST_TXT   = f'{BASE_PATH}/split/test.txt'
    IMG_DIR    = f'{BASE_PATH}/image'
    LBL_DIR    = f'{BASE_PATH}/label'
    TEST_DIR   = f'{BASE_PATH}/prediction/image'
    SAVE_DIR   = '/kaggle/working'

    # ── Channels ───────────────────────────────────────────────────────────
    # 6 raw + NDWI + MNDWI + SAR_ratio = 9 channels
    # Set USE_SPECTRAL_INDICES = False to go back to 6 channels
    USE_SPECTRAL_INDICES = True
    IN_CHANNELS = 9           # 6 raw + NDWI + MNDWI + HH/HV ratio
    NUM_CLASSES = 3
    IMG_SIZE    = 512

    # ── Training ───────────────────────────────────────────────────────────
    BATCH_SIZE       = 4
    EPOCHS           = 30
    LR               = 1e-4
    MIN_LR           = 1e-6
    WEIGHT_DECAY     = 1e-4
    GRAD_CLIP        = 1.0
    ACCUMULATE_STEPS = 4      # effective batch = 16
    PATIENCE         = 10

    # ── Class Weights ──────────────────────────────────────────────────────
    # Run Cell 6 (class balance check) and paste suggested values here.
    # These are reasonable defaults: flood gets 6x weight.
    CLASS_WEIGHTS = [1.0, 4.28, 3.35]

    # ── Inference ──────────────────────────────────────────────────────────
    TTA_ENABLED    = True
    # Flood threshold: tune on val set (Cell 15). Default 0.5 is usually suboptimal.
    FLOOD_THRESHOLD = 0.45    # lower = more flood pixels = higher recall
    # Min flood island size in pixels — removes noise
    MIN_FLOOD_PIXELS = 50

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Config ready ✅')

## 4. Data Loading

In [ ]:
def load_split(txt_path, img_dir, lbl_dir=None):
    ids = [l.strip() for l in open(txt_path) if l.strip()]
    imgs = [os.path.join(img_dir, f'{i}_image.tif') for i in ids]
    lbls = [os.path.join(lbl_dir, f'{i}_label.tif') for i in ids] if lbl_dir else None
    return imgs, lbls, ids

train_imgs, train_lbls, train_ids = load_split(CFG.TRAIN_TXT, CFG.IMG_DIR, CFG.LBL_DIR)
val_imgs,   val_lbls,   val_ids   = load_split(CFG.VAL_TXT,   CFG.IMG_DIR, CFG.LBL_DIR)

if os.path.exists(CFG.TEST_TXT):
    test_imgs, _, test_ids = load_split(CFG.TEST_TXT, CFG.TEST_DIR)
else:
    import glob
    test_imgs = sorted(glob.glob(os.path.join(CFG.TEST_DIR, '*.tif')))
    test_ids  = [os.path.basename(f).replace('_image.tif', '') for f in test_imgs]

print(f'Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}')

## 5. Per-Channel Stats (z-score normalization)

In [ ]:
def compute_channel_stats(img_paths, sample_n=50):
    sample = random.sample(img_paths, min(sample_n, len(img_paths)))
    all_means, all_stds = [], []
    for p in tqdm(sample, desc='Computing stats'):
        with rasterio.open(p) as f:
            img = f.read().astype(np.float32)          # (C, H, W)
        all_means.append(img.mean(axis=(1,2)))
        all_stds.append(img.std(axis=(1,2)))
    means = np.stack(all_means).mean(0)
    stds  = np.stack(all_stds).mean(0)
    stds  = np.where(stds < 1e-6, 1.0, stds)
    return means, stds

STATS_FILE = os.path.join(CFG.SAVE_DIR, 'channel_stats.npy')
if os.path.exists(STATS_FILE):
    stats = np.load(STATS_FILE, allow_pickle=True).item()
    MEANS, STDS = stats['means'], stats['stds']
    print('Loaded cached stats ✅')
else:
    MEANS, STDS = compute_channel_stats(train_imgs)
    np.save(STATS_FILE, {'means': MEANS, 'stds': STDS})
    print('Stats computed ✅')

print(f'Means: {MEANS.round(2)}')
print(f'Stds : {STDS.round(2)}')

## 6. Class Balance Check (run once, set CFG.CLASS_WEIGHTS)

In [ ]:
# Run this once to get optimal class weights
from tqdm import tqdm
class_counts = np.zeros(3, dtype=np.int64)
for p in tqdm(train_lbls, desc='Counting pixels'):
    with rasterio.open(p) as f:
        lbl = f.read(1).astype(int)
    for c in range(3):
        class_counts[c] += (lbl == c).sum()

total = class_counts.sum()
names = ['NoFlood', 'Flood', 'WaterBody']
print('Class distribution:')
for name, cnt in zip(names, class_counts):
    print(f'  {name}: {cnt:,} ({100*cnt/total:.2f}%)')

freqs = class_counts / total
inv_w = 1.0 / (freqs + 1e-8)
inv_w /= inv_w.min()
print('\nSuggested CLASS_WEIGHTS:', list(np.round(inv_w, 2)))
print('  → Set CFG.CLASS_WEIGHTS to these values')

## 7. Spectral Index Feature Engineering

In [ ]:
def add_spectral_indices(img):
    """
    Input:  img [6, H, W] raw float (HH, HV, Green, Red, NIR, SWIR)
    Output: img [9, H, W] — adds NDWI, MNDWI, SAR_ratio

    Why each matters:
    - NDWI  = (Green-NIR)/(Green+NIR): water positive, land negative
    - MNDWI = (Green-SWIR)/(Green+SWIR): suppresses built-up false positives
    - SAR_ratio = HH/HV: flooded vegetation shows high HH/HV ratio
      because double-bounce scattering spikes HH but not HV
    """
    hh, hv   = img[0], img[1]
    green    = img[2]
    nir      = img[4]
    swir     = img[5]
    eps      = 1e-8

    ndwi    = (green - nir)  / (green + nir  + eps)
    mndwi   = (green - swir) / (green + swir + eps)
    sar_rat = np.log1p(np.abs(hh)) - np.log1p(np.abs(hv))  # log ratio, stable

    # Clip to [-1,1] for ndwi/mndwi; sar_ratio is already in log space
    ndwi    = np.clip(ndwi,  -1, 1)
    mndwi   = np.clip(mndwi, -1, 1)

    return np.concatenate([img,
                           ndwi[np.newaxis],
                           mndwi[np.newaxis],
                           sar_rat[np.newaxis]], axis=0)  # [9, H, W]


def normalize_image(image, means, stds, clip_pct=2):
    """
    Per-channel: clip outlier percentiles → z-score.
    Called identically by train AND test datasets — single source of truth.
    """
    image = image.copy()
    for c in range(image.shape[0]):
        lo = np.percentile(image[c], clip_pct)
        hi = np.percentile(image[c], 100 - clip_pct)
        image[c] = np.clip(image[c], lo, hi)
        image[c] = (image[c] - means[c]) / (stds[c] + 1e-8)
    return image


def get_extended_stats(means, stds, sample_imgs, n=50):
    """
    Extend channel stats (mean/std) to cover spectral index channels.
    Needed because add_spectral_indices creates 3 new channels.
    """
    sample = random.sample(sample_imgs, min(n, len(sample_imgs)))
    ext_means, ext_stds = [], []
    for p in tqdm(sample, desc='Extended stats'):
        with rasterio.open(p) as f:
            img = f.read().astype(np.float32)
        img_aug = add_spectral_indices(img)   # [9, H, W]
        ext_means.append(img_aug.mean(axis=(1,2)))
        ext_stds.append(img_aug.std(axis=(1,2)))
    em = np.stack(ext_means).mean(0)
    es = np.stack(ext_stds).mean(0)
    es = np.where(es < 1e-6, 1.0, es)
    return em, es

if CFG.USE_SPECTRAL_INDICES:
    EXT_STATS_FILE = os.path.join(CFG.SAVE_DIR, 'ext_channel_stats.npy')
    if os.path.exists(EXT_STATS_FILE):
        s = np.load(EXT_STATS_FILE, allow_pickle=True).item()
        EXT_MEANS, EXT_STDS = s['means'], s['stds']
        print('Loaded extended stats ✅')
    else:
        EXT_MEANS, EXT_STDS = get_extended_stats(MEANS, STDS, train_imgs)
        np.save(EXT_STATS_FILE, {'means': EXT_MEANS, 'stds': EXT_STDS})
        print('Extended stats computed ✅')
    ACTIVE_MEANS, ACTIVE_STDS = EXT_MEANS, EXT_STDS
else:
    ACTIVE_MEANS, ACTIVE_STDS = MEANS, STDS
    print('Using raw 6-channel stats')

In [ ]:
class FloodDataset(Dataset):
    def __init__(self, imgs, lbls=None, transform=None,
                 means=None, stds=None):
        self.imgs      = imgs
        self.lbls      = lbls
        self.transform = transform
        self.means     = means if means is not None else ACTIVE_MEANS
        self.stds      = stds  if stds  is not None else ACTIVE_STDS

    def __len__(self): return len(self.imgs)

    def _load_image(self, path):
        with rasterio.open(path) as f:
            img = f.read().astype(np.float32)   # (C, H, W)
        img = img[:, :CFG.IMG_SIZE, :CFG.IMG_SIZE]
        if CFG.USE_SPECTRAL_INDICES:
            img = add_spectral_indices(img)     # (9, H, W)
        img = normalize_image(img, self.means, self.stds)
        return img

    def __getitem__(self, idx):
        image = self._load_image(self.imgs[idx])  # (C, H, W)
        image = np.transpose(image, (1, 2, 0))    # (H, W, C) for albumentations

        if self.lbls is not None:
            with rasterio.open(self.lbls[idx]) as f:
                label = f.read(1).astype(np.int64)
            label = label[:CFG.IMG_SIZE, :CFG.IMG_SIZE]
            label = np.clip(label, 0, CFG.NUM_CLASSES - 1)
        else:
            label = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE), dtype=np.int64)

        if self.transform:
            aug   = self.transform(image=image, mask=label)
            image = aug['image']     # (C, H, W) tensor
            label = aug['mask']      # (H, W) tensor
        else:
            image = torch.tensor(np.transpose(image, (2,0,1)), dtype=torch.float32)
            label = torch.tensor(label, dtype=torch.long)

        return image, label

print('Dataset defined ✅')

## 8. Dataset

In [ ]:
train_transform = A.Compose([
    # Geometric
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.85,1.15), translate_percent=(-0.08,0.08),
             rotate=(-45,45), shear=(-5,5), p=0.5),
    A.ElasticTransform(alpha=1, sigma=50, p=0.2),
    A.GridDistortion(p=0.2),
    # Photometric
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.3),  # SAR speckle simulation
    A.Blur(blur_limit=3, p=0.15),
    # Cutout — 0.0 fill = channel mean after z-score
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32,
                    min_holes=1, fill_value=0.0, mask_fill_value=0, p=0.2),
    ToTensorV2(),
])

val_transform = A.Compose([ToTensorV2()])

print('Augmentations defined ✅')

## 9. Augmentation Pipeline

In [ ]:
train_ds = FloodDataset(train_imgs, train_lbls, transform=train_transform)
val_ds   = FloodDataset(val_imgs,   val_lbls,   transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True,
                          persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True, persistent_workers=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## 10. DataLoaders

In [ ]:
def build_model_1():
    """UNet++ EfficientNet-B5 + scSE attention — strong multi-scale features."""
    return smp.UnetPlusPlus(
        encoder_name='efficientnet-b5',
        encoder_weights='imagenet',
        in_channels=CFG.IN_CHANNELS,
        classes=CFG.NUM_CLASSES,
        activation=None,
        decoder_attention_type='scse',
    )

def build_model_2():
    """DeepLabV3+ ResNet50 — strong at boundaries and thin flood filaments."""
    return smp.DeepLabV3Plus(
        encoder_name='resnet50',
        encoder_weights='imagenet',
        in_channels=CFG.IN_CHANNELS,
        classes=CFG.NUM_CLASSES,
        activation=None,
        encoder_output_stride=16,
    )

def build_model_3():
    """
    UNet mit-b3 (SegFormer encoder) — transformer backbone.
    Captures global context that CNNs miss — especially useful for
    large homogeneous flood plains.
    """
    return smp.Unet(
        encoder_name='mit_b3',
        encoder_weights='imagenet',
        in_channels=CFG.IN_CHANNELS,
        classes=CFG.NUM_CLASSES,
        activation=None,
    )

model1 = build_model_1().to(CFG.DEVICE)
model2 = build_model_2().to(CFG.DEVICE)
model3 = build_model_3().to(CFG.DEVICE)

for name, m in [('UNet++ EffB5', model1), ('DeepLabV3+ R50', model2), ('Unet MiT-B3', model3)]:
    n = sum(p.numel() for p in m.parameters()) / 1e6
    print(f'{name}: {n:.1f}M params')

## 11. Model Definitions (3-Model Ensemble)

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        targets = targets.long()
        preds = torch.softmax(preds, dim=1)
        targets_oh = F.one_hot(targets, num_classes=CFG.NUM_CLASSES).permute(0,3,1,2).float()
        inter = (preds * targets_oh).sum(dim=(2,3))
        union = preds.sum(dim=(2,3)) + targets_oh.sum(dim=(2,3))
        dice  = (2 * inter + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, preds, targets):
        targets = targets.long()
        log_p = F.log_softmax(preds, dim=1)
        ce    = F.nll_loss(log_p, targets, weight=self.weight, reduction='none')
        pt    = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


class CombinedLoss(nn.Module):
    def __init__(self, dice_w=0.5, focal_w=0.5, class_weights=CFG.CLASS_WEIGHTS):
        super().__init__()
        w = torch.tensor(class_weights, dtype=torch.float32, device=CFG.DEVICE)
        self.dice  = DiceLoss()              # ← was DiceLoss(class_weights=...) — wrong arg
        self.focal = FocalLoss(gamma=2.0, weight=w)
        self.dw    = dice_w
        self.fw    = focal_w

    def forward(self, preds, targets):
        targets = targets.long()
        return self.dw * self.dice(preds, targets) + self.fw * self.focal(preds, targets)


criterion = CombinedLoss()
print('Loss defined ✅')

In [ ]:
import os
import numpy as np
import torch

class CFG:
    # ── Paths ──────────────────────────────────────────────────────────────
    BASE_PATH  = '/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data'
    TRAIN_TXT  = f'{BASE_PATH}/split/train.txt'
    VAL_TXT    = f'{BASE_PATH}/split/val.txt'
    TEST_TXT   = f'{BASE_PATH}/split/test.txt'
    IMG_DIR    = f'{BASE_PATH}/image'
    LBL_DIR    = f'{BASE_PATH}/label'
    TEST_DIR   = f'{BASE_PATH}/prediction/image'
    SAVE_DIR   = '/kaggle/working'

    USE_SPECTRAL_INDICES = True
    IN_CHANNELS = 9
    NUM_CLASSES = 3
    IMG_SIZE    = 512

    BATCH_SIZE       = 4
    EPOCHS           = 60
    LR               = 1e-4
    MIN_LR           = 1e-6
    WEIGHT_DECAY     = 1e-4
    GRAD_CLIP        = 1.0
    ACCUMULATE_STEPS = 4
    PATIENCE         = 10

    CLASS_WEIGHTS    = [1.0, 6.0, 2.0]

    TTA_ENABLED      = True
    FLOOD_THRESHOLD  = 0.45
    MIN_FLOOD_PIXELS = 50

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


class SegMetrics:
    def __init__(self, n=CFG.NUM_CLASSES):
        self.n = n
        self.reset()

    def reset(self):
        self.cm = np.zeros((self.n, self.n), dtype=np.int64)

    def update(self, preds, targets):
        preds   = preds.argmax(1).cpu().numpy().flatten()
        targets = targets.cpu().numpy().flatten()
        valid   = (targets >= 0) & (targets < self.n)
        idx     = self.n * targets[valid] + preds[valid]
        self.cm += np.bincount(idx, minlength=self.n**2).reshape(self.n, self.n)

    def compute(self):
        tp  = np.diag(self.cm)
        fn  = self.cm.sum(1) - tp
        fp  = self.cm.sum(0) - tp
        iou = np.where(tp+fn+fp > 0, tp / (tp+fn+fp+1e-8), 0.0)
        return {
            'iou_noflood' : iou[0],
            'iou_flood'   : iou[1],
            'iou_water'   : iou[2],
            'mIoU'        : iou.mean(),
            'acc'         : tp.sum() / self.cm.sum()
        }

print('SegMetrics defined ✅')

## 12. Loss Functions

In [ ]:
class SegMetrics:
    def __init__(self, n=CFG.NUM_CLASSES):
        self.n = n
        self.reset()

    def reset(self):
        self.cm = np.zeros((self.n, self.n), dtype=np.int64)

    def update(self, preds, targets):
        preds   = preds.argmax(1).cpu().numpy().flatten()
        targets = targets.cpu().numpy().flatten()
        valid   = (targets >= 0) & (targets < self.n)
        idx     = self.n * targets[valid] + preds[valid]
        self.cm += np.bincount(idx, minlength=self.n**2).reshape(self.n, self.n)

    def compute(self):
        tp  = np.diag(self.cm)
        fn  = self.cm.sum(1) - tp
        fp  = self.cm.sum(0) - tp
        iou = np.where(tp+fn+fp > 0, tp / (tp+fn+fp+1e-8), 0.0)
        return {'iou_noflood': iou[0], 'iou_flood': iou[1],
                'iou_water': iou[2], 'mIoU': iou.mean(),
                'acc': tp.sum() / self.cm.sum()}

print('Metrics defined ✅')

## 13. Metrics

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, (imgs, masks) in enumerate(loader):
        imgs  = imgs.to(CFG.DEVICE)
        masks = masks.to(CFG.DEVICE).long()   # ← enforce long here too

        use_amp = (CFG.DEVICE == 'cuda')
        with torch.amp.autocast(CFG.DEVICE, enabled=use_amp):
            loss = criterion(model(imgs), masks) / CFG.ACCUMULATE_STEPS

        if use_amp:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (step + 1) % CFG.ACCUMULATE_STEPS == 0:
            if use_amp:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
                optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * CFG.ACCUMULATE_STEPS

    scheduler.step()
    return total_loss / len(loader)

## 14. Training Loop

In [ ]:
import torch
import gc

# ── Build models fresh (safe to re-run after any kernel restart) ───────────
def build_model_1():
    return smp.UnetPlusPlus(
        encoder_name='efficientnet-b5',
        encoder_weights='imagenet',
        in_channels=CFG.IN_CHANNELS,
        classes=CFG.NUM_CLASSES,
        activation=None,
        decoder_attention_type='scse',
    )

def build_model_2():
    return smp.DeepLabV3Plus(
        encoder_name='resnet50',
        encoder_weights='imagenet',
        in_channels=CFG.IN_CHANNELS,
        classes=CFG.NUM_CLASSES,
        activation=None,
        encoder_output_stride=16,
    )

def build_model_3():
    return smp.Unet(
        encoder_name='mit_b3',
        encoder_weights='imagenet',
        in_channels=CFG.IN_CHANNELS,
        classes=CFG.NUM_CLASSES,
        activation=None,
    )

model1 = build_model_1().to(CFG.DEVICE)
model2 = build_model_2().to(CFG.DEVICE)
model3 = build_model_3().to(CFG.DEVICE)

for name, m in [('UNet++ EffB5', model1), ('DeepLabV3+ R50', model2), ('Unet MiT-B3', model3)]:
    n = sum(p.numel() for p in m.parameters()) / 1e6
    print(f'{name}: {n:.1f}M params')

print('\nModels ready ✅')

# ── Train ──────────────────────────────────────────────────────────────────
history1 = train_model(model1, 'UNet++ EfficientNet-B5', 'best_model1.pth')
torch.cuda.empty_cache(); gc.collect()

history2 = train_model(model2, 'DeepLabV3+ ResNet50', 'best_model2.pth')
torch.cuda.empty_cache(); gc.collect()

history3 = train_model(model3, 'Unet MiT-B3', 'best_model3.pth')
torch.cuda.empty_cache(); gc.collect()

In [ ]:
for name, hist in [
    ('UNet++ EffB5', history1),
    ('DeepLabV3+ R50', history2),
    ('Unet MiT-B3', history3),
]:
    # hist is a dict: {'train_loss': [...], 'val_metrics': [ {...}, {...}, ... ]}
    val_hist = hist['val_metrics']

    # add epoch index into each record if not already present
    for i, rec in enumerate(val_hist):
        if 'epoch' not in rec:
            rec['epoch'] = i + 1

    best = max(val_hist, key=lambda x: x['iou_flood'])
    print(
        f"{name}: flood_IoU={best['iou_flood']:.4f}, "
        f"mIoU={best['mIoU']:.4f}, ep={best['epoch']:.0f}"
    )

In [ ]:
"""
WHY: argmax(logits) implicitly uses threshold=0.5 for the flood class.
Because flood is rare, the model is conservative. Lowering the flood
probability threshold recovers missed flood pixels and boosts recall
→ flood IoU typically improves by 1-3 points.
"""

# Load best model 1 for threshold scan
m = build_model_1().to(CFG.DEVICE)

ckpt = torch.load(
    os.path.join(CFG.SAVE_DIR, 'best_model1.pth'),
    map_location=CFG.DEVICE
)

# If checkpoint is saved as plain state_dict
if isinstance(ckpt, dict) and 'state_dict' not in ckpt:
    m.load_state_dict(ckpt)
# If checkpoint is saved as {'state_dict': ...}
elif isinstance(ckpt, dict) and 'state_dict' in ckpt:
    m.load_state_dict(ckpt['state_dict'])
else:
    raise ValueError("Unknown checkpoint format for best_model1.pth")

m.eval()

# Collect all softmax probs and ground truth on val set
all_probs, all_gt = [], []

with torch.no_grad():
    for imgs, masks in tqdm(val_loader, desc='Collecting probs'):
        imgs = imgs.to(CFG.DEVICE)

        use_amp = (CFG.DEVICE == 'cuda')
        with torch.amp.autocast(CFG.DEVICE, enabled=use_amp):
            logits = m(imgs)

        probs = torch.softmax(logits, dim=1).cpu().numpy()   # (B, 3, H, W)
        all_probs.append(probs)
        all_gt.append(masks.numpy())

all_probs = np.concatenate(all_probs, axis=0)   # (N, 3, H, W)
all_gt    = np.concatenate(all_gt, axis=0)      # (N, H, W)

# Grid search threshold
flood_probs = all_probs[:, 1]   # flood channel probability
water_probs = all_probs[:, 2]   # water channel probability

best_t, best_iou = 0.5, 0.0

for t in np.arange(0.30, 0.65, 0.025):
    pred = all_probs.argmax(axis=1)   # baseline argmax

    # Override prediction: make pixel flood if flood prob crosses threshold
    # and is stronger than water probability
    flood_wins = (flood_probs > t) & (flood_probs > water_probs)

    pred_t = pred.copy()
    pred_t[flood_wins] = 1

    # Compute flood IoU
    tp = ((pred_t == 1) & (all_gt == 1)).sum()
    fp = ((pred_t == 1) & (all_gt != 1)).sum()
    fn = ((pred_t != 1) & (all_gt == 1)).sum()

    iou = tp / (tp + fp + fn + 1e-8)
    print(f't={t:.3f} → flood_IoU={iou:.4f}')

    if iou > best_iou:
        best_iou = iou
        best_t = t

print(f'\nBest threshold: {best_t:.3f} → flood_IoU={best_iou:.4f}')
print(f'Set CFG.FLOOD_THRESHOLD = {best_t:.3f}')

CFG.FLOOD_THRESHOLD = best_t

del m, ckpt
torch.cuda.empty_cache()
gc.collect()

In [ ]:
def load_best(build_fn, ckpt_name):
    m = build_fn().to(CFG.DEVICE)
    ckpt_path = os.path.join(CFG.SAVE_DIR, ckpt_name)
    ckpt = torch.load(ckpt_path, map_location=CFG.DEVICE)

    # Handle both possible formats safely
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        # case: torch.save({'state_dict': model.state_dict(), ...})
        m.load_state_dict(ckpt['state_dict'])
    else:
        # case: torch.save(model.state_dict())
        m.load_state_dict(ckpt)

    m.eval()
    return m


model1 = load_best(build_model_1, 'best_model1.pth')
model2 = load_best(build_model_2, 'best_model2.pth')
model3 = load_best(build_model_3, 'best_model3.pth')

# Weights: tune based on val flood IoU scores
ENSEMBLE_WEIGHTS = [0.40, 0.30, 0.30]
print(f'Models loaded ✅. Ensemble weights: {ENSEMBLE_WEIGHTS}')

In [ ]:
def tta_transform(img, k):
    """Apply one of 8 deterministic TTA variants. img: (C,H,W) numpy."""
    if k == 0: return img
    if k == 1: return np.flip(img, axis=2).copy()
    if k == 2: return np.flip(img, axis=1).copy()
    if k == 3: return np.rot90(img, 1, (1,2)).copy()
    if k == 4: return np.rot90(img, 2, (1,2)).copy()
    if k == 5: return np.rot90(img, 3, (1,2)).copy()
    if k == 6: return np.rot90(np.flip(img, axis=2), 1, (1,2)).copy()
    if k == 7: return np.rot90(np.flip(img, axis=1), 1, (1,2)).copy()

def tta_reverse(pred, k):
    """Reverse TTA transform on prediction logits. pred: (C,H,W) numpy."""
    if k == 0: return pred
    if k == 1: return np.flip(pred, axis=2).copy()
    if k == 2: return np.flip(pred, axis=1).copy()
    if k == 3: return np.rot90(pred, -1, (1,2)).copy()
    if k == 4: return np.rot90(pred, -2, (1,2)).copy()
    if k == 5: return np.rot90(pred, -3, (1,2)).copy()
    if k == 6:
        p = np.rot90(pred, -1, (1,2))
        return np.flip(p, axis=2).copy()
    if k == 7:
        p = np.rot90(pred, -1, (1,2))
        return np.flip(p, axis=1).copy()


@torch.no_grad()
def predict_single(model, img_np, tta_ids):
    """
    img_np: (C, H, W) normalized numpy.
    Returns (C, H, W) averaged softmax probabilities.
    """
    model.eval()
    probs_acc = np.zeros((CFG.NUM_CLASSES, CFG.IMG_SIZE, CFG.IMG_SIZE), np.float32)
    for k in tta_ids:
        aug = tta_transform(img_np, k)
        x   = torch.from_numpy(aug).float().unsqueeze(0).to(CFG.DEVICE)
        with torch.amp.autocast('cuda'):
            logits = model(x)
        p = torch.softmax(logits.squeeze(0), dim=0).cpu().numpy()
        probs_acc += tta_reverse(p, k)
    return probs_acc / len(tta_ids)

print('TTA engine defined ✅')

In [ ]:
def postprocess_flood_mask(mask, min_pixels=CFG.MIN_FLOOD_PIXELS):
    """
    Morphological post-processing:
    1. Remove noise islands < min_pixels px (connected component filtering)
    2. Fill small holes within flood regions (binary closing)

    Why this helps: models predict isolated 1-5 pixel noise blobs that are
    physically impossible as flood. Removing them reduces false positives
    without hurting recall on real flood patches.
    """
    if mask.sum() == 0:
        return mask

    # Label connected components
    labeled, num = ndimage.label(mask)
    sizes = ndimage.sum(mask, labeled, range(1, num + 1))

    # Keep only large enough components
    clean = np.zeros_like(mask)
    for i, size in enumerate(sizes):
        if size >= min_pixels:
            clean[labeled == (i + 1)] = 1

    # Binary closing to fill small holes (3x3 structuring element)
    struct = ndimage.generate_binary_structure(2, 1)
    clean  = ndimage.binary_closing(clean, structure=struct, iterations=2).astype(np.uint8)

    return clean

print('Post-processing defined ✅')

In [ ]:
def rle_encode(mask):
    """Column-major RLE (Kaggle convention). Returns '0 0' for empty mask."""
    mask = mask.astype(np.uint8)
    if mask.sum() == 0:
        return '0 0'
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(map(str, runs))


def rle_decode(rle_str, shape=(512, 512)):
    if rle_str in ['0 0', '', None]:
        return np.zeros(shape, dtype=np.uint8)
    nums    = list(map(int, rle_str.split()))
    starts  = np.array(nums[0::2]) - 1
    lengths = np.array(nums[1::2])
    flat    = np.zeros(shape[0]*shape[1], dtype=np.uint8)
    for s, l in zip(starts, lengths):
        flat[s:s+l] = 1
    return flat.reshape(shape, order='F')

print('RLE defined ✅')

In [ ]:
# Rebuild test list directly from prediction/image folder
test_dir = CFG.TEST_DIR  # should be .../data/prediction/image

all_files = sorted(os.listdir(test_dir))
# Keep only tif files
tif_files = [f for f in all_files if f.endswith('.tif')]

test_imgs = [os.path.join(test_dir, f) for f in tif_files]
# Derive id from filename: remove '_image.tif'
test_ids  = [f.replace('_image.tif', '') for f in tif_files]

print("Found test images:")
for p, i in zip(test_imgs, test_ids):
    print(i, "->", p)

print(f"\nTotal test images: {len(test_imgs)}")

In [ ]:
"""
CRITICAL FIX vs v2.1:
The old notebook had TestDataset using cv2.imread() + /255 normalization.
This is COMPLETELY WRONG for multi-channel SAR GeoTIFFs because:
  1. cv2.imread only reads 3 channels (BGR)
  2. /255 normalization doesn't match the per-channel z-score used in training
This alone likely explains most of the rank gap.

Fix: use rasterio.open() + normalize_image() — same as FloodDataset.
"""

TTA_IDS = list(range(8)) if CFG.TTA_ENABLED else [0]
submission = []

print(f'Inference on {len(test_imgs)} images | TTA variants: {len(TTA_IDS)}')

missing = []

for i, (img_path, img_id) in enumerate(tqdm(zip(test_imgs, test_ids), total=len(test_imgs))):

    # Skip if file does not actually exist (defensive check)
    if not os.path.exists(img_path):
        print(f"[WARN] Missing file for id={img_id}: {img_path}")
        missing.append(img_id)
        # Append empty mask (all background) to keep submission length consistent
        dummy_mask = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE), dtype=np.uint8)
        submission.append({'id': img_id, 'rle_mask': rle_encode(dummy_mask)})
        continue

    # ── Load & preprocess (SAME as training) ──────────────────────────────
    with rasterio.open(img_path) as f:
        img_raw = f.read().astype(np.float32)          # (6, H, W)

    img_raw = img_raw[:, :CFG.IMG_SIZE, :CFG.IMG_SIZE]

    if CFG.USE_SPECTRAL_INDICES:
        img_raw = add_spectral_indices(img_raw)        # (9, H, W)

    img_norm = normalize_image(img_raw, ACTIVE_MEANS, ACTIVE_STDS)  # (9, H, W)

    # ── Multi-model TTA ensemble ───────────────────────────────────────────
    prob1 = predict_single(model1, img_norm, TTA_IDS)
    prob2 = predict_single(model2, img_norm, TTA_IDS)
    prob3 = predict_single(model3, img_norm, TTA_IDS)

    ensemble = (ENSEMBLE_WEIGHTS[0] * prob1 +
                ENSEMBLE_WEIGHTS[1] * prob2 +
                ENSEMBLE_WEIGHTS[2] * prob3)          # (3, H, W)

    # ── Thresholded prediction ─────────────────────────────────────────────
    flood_prob = ensemble[1]                           # class 1 prob
    water_prob = ensemble[2]                           # class 2 prob

    # Base prediction via argmax
    pred = ensemble.argmax(axis=0)

    # Override with tuned threshold: flood wins if its prob > threshold AND > water
    flood_wins = (flood_prob > CFG.FLOOD_THRESHOLD) & (flood_prob > water_prob)
    pred[flood_wins] = 1

    # ── Post-processing ────────────────────────────────────────────────────
    flood_mask = (pred == 1).astype(np.uint8)
    flood_mask = postprocess_flood_mask(flood_mask)

    # ── RLE encode ────────────────────────────────────────────────────────
    submission.append({'id': img_id, 'rle_mask': rle_encode(flood_mask)})

    if (i + 1) % 100 == 0:
        gc.collect()

print(f'Inference complete ✅ ({len(submission)} predictions)')
if missing:
    print(f'Missing {len(missing)} files, filled with empty masks: {missing}')

In [ ]:
df = pd.DataFrame(submission)

assert len(df) == len(test_imgs),      '❌ Missing test images'
assert df['id'].nunique() == len(df),  '❌ Duplicate IDs'
assert df['rle_mask'].isna().sum() == 0, '❌ Null values'
assert (df['rle_mask'] == '').sum() == 0, '❌ Empty strings'

n_flood = (df['rle_mask'] != '0 0').sum()
print(f'Total: {len(df)} | Flood: {n_flood} ({100*n_flood/len(df):.1f}%) | NoFlood: {len(df)-n_flood}')
print(df.head(5).to_string())

out_path = os.path.join(CFG.SAVE_DIR, 'submission_v3.csv')
df.to_csv(out_path, index=False)
print(f'\n🚀 Saved → {out_path}')

In [ ]:
for row in df.sample(min(5, len(df))).itertuples():
    if row.rle_mask == '0 0':
        print(f'{row.id}: empty mask'); continue
    dec   = rle_decode(row.rle_mask)
    match = rle_encode(dec) == row.rle_mask
    print(f'{row.id}: flood_px={dec.sum()}, round-trip={"✅" if match else "❌"}')

print('\n✅ All checks passed. Ready for submission!')

In [ ]:
# ── Pseudo-labeling: add high-confidence test images to train set ──────────
# Only use predictions where ALL three models agree AND max prob > 0.90

pseudo_imgs, pseudo_lbls = [], []
CONFIDENCE_THRESHOLD = 0.90

print('Generating pseudo-labels from test set...')
for img_path, img_id in tqdm(zip(test_imgs, test_ids), total=len(test_imgs)):
    with rasterio.open(img_path) as f:
        img_raw = f.read().astype(np.float32)
    img_raw  = img_raw[:, :CFG.IMG_SIZE, :CFG.IMG_SIZE]
    if CFG.USE_SPECTRAL_INDICES:
        img_raw = add_spectral_indices(img_raw)
    img_norm = normalize_image(img_raw, ACTIVE_MEANS, ACTIVE_STDS)

    prob1 = predict_single(model1, img_norm, [0])  # no TTA for speed
    prob2 = predict_single(model2, img_norm, [0])
    prob3 = predict_single(model3, img_norm, [0])
    ens   = (prob1 + prob2 + prob3) / 3

    max_prob = ens.max(axis=0)       # (H, W)
    pred_lbl = ens.argmax(axis=0)    # (H, W)

    # Only keep image if most pixels are high-confidence
    high_conf_frac = (max_prob > CONFIDENCE_THRESHOLD).mean()
    if high_conf_frac > 0.85:        # 85% of pixels must be confident
        # Save pseudo-label as npy
        pseudo_path = os.path.join(CFG.SAVE_DIR, f'pseudo_{img_id}.npy')
        np.save(pseudo_path, pred_lbl.astype(np.uint8))
        pseudo_imgs.append(img_path)
        pseudo_lbls.append(pseudo_path)

print(f'High-confidence pseudo-labels: {len(pseudo_imgs)}/{len(test_imgs)}')
print('  → Re-run training with: train_imgs + pseudo_imgs as combined dataset')
print('  → Use lower LR (5e-5) and fewer epochs (20) for Round 2')